# Fase 1 - Densidad de datos de accesibilidad en Madrid

Este notebook analiza la cobertura de OpenStreetMap y Mapillary en Madrid capital para decidir una zona piloto del MVP de rutas accesibles.

Objetivos:

- Medir disponibilidad de atributos de accesibilidad por zona.
- Separar datos estructurados OSM de cobertura visual Mapillary.
- Identificar atributos disponibles, ausentes e inciertos.
- Generar mapas de densidad y CSVs reproducibles.
- Proponer zonas candidatas para el area piloto.

Notas de seguridad del TFM:

- La ausencia de datos no implica ausencia de barreras.
- Este analisis mide cobertura de datos, no accesibilidad real.
- Los atributos desconocidos deben reducir confianza o generar aviso en el MVP.


## 0. Preparacion

Requisitos recomendados desde la raiz del repo:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m ipykernel install --user --name tfm-density --display-name "TFM density"
```

Mapillary requiere token. Crear un `.env` local, no versionado:

```text
MAPILLARY_ACCESS_TOKEN=...
```

Si no hay token, el notebook ejecuta la parte OSM y deja Mapillary marcado como no disponible. Por defecto Mapillary descarga hasta 5 paginas para evitar llamadas masivas; se puede ajustar con `MAPILLARY_MAX_PAGES` en `.env`.


In [ ]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv
from folium.features import GeoJsonTooltip
from shapely.geometry import Point, box
from tqdm.auto import tqdm

load_dotenv()

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
MAPS_DIR = REPO_ROOT / "outputs" / "maps"
for directory in (DATA_RAW, DATA_PROCESSED, MAPS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

WGS84 = "EPSG:4326"
MADRID_CRS = "EPSG:25830"  # ETRS89 / UTM zone 30N, suitable for Madrid distances.

# Madrid capital approximate bounding box: south, west, north, east.
MADRID_BBOX = (40.3121, -3.8342, 40.6437, -3.5249)
MADRID_CENTER = (40.4168, -3.7038)
GRID_SIZE_METERS = 1000

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
MAPILLARY_TOKEN = os.getenv("MAPILLARY_ACCESS_TOKEN")
MAPILLARY_MAX_PAGES = int(os.getenv("MAPILLARY_MAX_PAGES", "5"))

pd.set_option("display.max_columns", 80)


## 1. Atributos de accesibilidad a medir

Los atributos se eligen porque son relevantes para rutas de personas ciegas o con baja vision y porque existen como senales observables o tags OSM verificables. No se inventan tags: si un atributo no aparece en OSM, se marca como ausente/desconocido.


In [ ]:
@dataclass(frozen=True)
class OsmAttributeQuery:
    name: str
    description: str
    overpass_selector: str
    safety_note: str


OSM_ATTRIBUTE_QUERIES = [
    OsmAttributeQuery(
        "crossings",
        "Pasos de peatones etiquetados como highway=crossing.",
        'nwr["highway"="crossing"]',
        "La presencia de cruce no confirma que sea seguro ni accesible.",
    ),
    OsmAttributeQuery(
        "traffic_signals",
        "Semaforos etiquetados como highway=traffic_signals.",
        'nwr["highway"="traffic_signals"]',
        "No todos los semaforos tienen senal acustica.",
    ),
    OsmAttributeQuery(
        "audible_signals",
        "Semaforos o cruces con senal acustica/vibracion declarada.",
        'nwr["traffic_signals:sound"];nwr["traffic_signals:vibration"]',
        "Atributo critico: la ausencia de tag no implica ausencia real.",
    ),
    OsmAttributeQuery(
        "tactile_paving",
        "Pavimento tactil declarado.",
        'nwr["tactile_paving"]',
        "Atributo muy incompleto en muchas ciudades.",
    ),
    OsmAttributeQuery(
        "kerbs",
        "Bordillos, rebajes o kerbs declarados.",
        'nwr["kerb"]',
        "La semantica del valor debe analizarse antes de puntuar accesibilidad.",
    ),
    OsmAttributeQuery(
        "sidewalks",
        "Calles o vias con atributo sidewalk.",
        'way["sidewalk"]',
        "Sidewalk=yes/left/right/no requiere interpretacion por tramo.",
    ),
    OsmAttributeQuery(
        "ramps_or_wheelchair",
        "Rampas o accesibilidad wheelchair declarada.",
        'nwr["ramp"];nwr["wheelchair"]',
        "No debe asumirse que wheelchair=yes equivale a ruta segura para todos.",
    ),
    OsmAttributeQuery(
        "steps",
        "Escaleras etiquetadas como highway=steps.",
        'nwr["highway"="steps"]',
        "Puede ser barrera critica segun perfil del usuario.",
    ),
    OsmAttributeQuery(
        "surface_on_pedestrian_ways",
        "Superficie declarada en vias peatonales o calles relevantes.",
        'way["highway"~"^(footway|path|pedestrian|steps|living_street|residential|service|unclassified|tertiary|secondary|primary)$"]["surface"]',
        "La superficie puede afectar seguridad, pero los valores deben normalizarse.",
    ),
    OsmAttributeQuery(
        "incline",
        "Pendiente declarada mediante incline.",
        'nwr["incline"]',
        "Dato escaso; para produccion convendria combinar con DEM.",
    ),
]

pd.DataFrame([query.__dict__ for query in OSM_ATTRIBUTE_QUERIES])


## 2. Malla de analisis

Se usa una malla regular de 1 km sobre el bounding box de Madrid capital. Esto evita depender de limites administrativos externos y permite comparar cobertura espacial de forma reproducible. Las celdas son zonas candidatas; despues se puede cruzar con distritos/barrios oficiales si se anade una fuente fiable.


In [ ]:
def build_grid(bbox_south_west_north_east: tuple[float, float, float, float], cell_size_m: int) -> gpd.GeoDataFrame:
    """Build a regular grid over a WGS84 bounding box.

    Args:
        bbox_south_west_north_east: Tuple with south, west, north, east coordinates.
        cell_size_m: Cell size in meters in the projected CRS.

    Returns:
        GeoDataFrame with one polygon per analysis zone.
    """
    south, west, north, east = bbox_south_west_north_east
    bbox_gdf = gpd.GeoDataFrame(geometry=[box(west, south, east, north)], crs=WGS84).to_crs(MADRID_CRS)
    minx, miny, maxx, maxy = bbox_gdf.total_bounds

    cells = []
    y = miny
    row = 0
    while y < maxy:
        x = minx
        col = 0
        while x < maxx:
            cells.append({"zone_id": f"z{row:02d}_{col:02d}", "geometry": box(x, y, x + cell_size_m, y + cell_size_m)})
            x += cell_size_m
            col += 1
        y += cell_size_m
        row += 1

    grid = gpd.GeoDataFrame(cells, crs=MADRID_CRS)
    grid["area_km2"] = grid.geometry.area / 1_000_000
    return grid.to_crs(WGS84)


grid = build_grid(MADRID_BBOX, GRID_SIZE_METERS)
grid.head()


In [ ]:
grid.explore(
    tiles="CartoDB positron",
    tooltip=["zone_id", "area_km2"],
    style_kwds={"fillOpacity": 0.05, "weight": 0.5},
)


## 3. Descarga reproducible de OpenStreetMap via Overpass

Cada consulta se guarda en `data/raw/osm_<atributo>.json` para poder reanalizar sin repetir llamadas. Se usa `out center tags` para estimar densidad espacial con puntos representativos; este notebook no pretende reconstruir geometrias completas de routing.


In [ ]:
def overpass_query(attribute: OsmAttributeQuery, bbox: tuple[float, float, float, float]) -> dict[str, Any]:
    """Run or load an Overpass query for one accessibility attribute.

    Args:
        attribute: Attribute query definition.
        bbox: Tuple with south, west, north, east coordinates.

    Returns:
        Raw Overpass JSON response.
    """
    cache_path = DATA_RAW / f"osm_{attribute.name}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    south, west, north, east = bbox
    selector_lines = []
    for selector in attribute.overpass_selector.split(";"):
        selector = selector.strip()
        if selector:
            selector_lines.append(f"  {selector}({south},{west},{north},{east});")

    query = "\n".join([
        "[out:json][timeout:180];",
        "(",
        *selector_lines,
        ");",
        "out center tags;",
    ])

    response = requests.post(OVERPASS_URL, data={"data": query}, timeout=240)
    response.raise_for_status()
    payload = response.json()
    cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    time.sleep(2)  # Be gentle with Overpass.
    return payload


def osm_elements_to_points(payload: dict[str, Any], attribute_name: str) -> gpd.GeoDataFrame:
    """Convert Overpass elements to representative points.

    Args:
        payload: Raw Overpass JSON response.
        attribute_name: Name assigned to all returned elements.

    Returns:
        GeoDataFrame with representative points and OSM tags.
    """
    rows = []
    for element in payload.get("elements", []):
        lon = element.get("lon")
        lat = element.get("lat")
        if lon is None or lat is None:
            center = element.get("center") or {}
            lon = center.get("lon")
            lat = center.get("lat")
        if lon is None or lat is None:
            continue
        rows.append({
            "source": "osm",
            "attribute": attribute_name,
            "osm_type": element.get("type"),
            "osm_id": element.get("id"),
            "tags": element.get("tags", {}),
            "geometry": Point(lon, lat),
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)


In [ ]:
osm_layers = []
for attribute in tqdm(OSM_ATTRIBUTE_QUERIES, desc="OSM attributes"):
    payload = overpass_query(attribute, MADRID_BBOX)
    layer = osm_elements_to_points(payload, attribute.name)
    print(f"{attribute.name}: {len(layer):,} elements")
    osm_layers.append(layer)

osm_points = pd.concat(osm_layers, ignore_index=True) if osm_layers else gpd.GeoDataFrame(geometry=[], crs=WGS84)
osm_points = gpd.GeoDataFrame(osm_points, geometry="geometry", crs=WGS84)
osm_points["tags_json"] = osm_points["tags"].apply(lambda tags: json.dumps(tags, ensure_ascii=False, sort_keys=True))
osm_points.drop(columns=["tags"]).to_parquet(DATA_PROCESSED / "osm_accessibility_points.parquet", index=False)
osm_points.head()


## 4. Cobertura visual de Mapillary

Mapillary se usa aqui como proxy de cobertura visual disponible, no como fuente de verdad de accesibilidad. La deteccion automatica de atributos mediante VLM queda para una fase final.


In [ ]:
def fetch_mapillary_images(bbox: tuple[float, float, float, float], token: str | None, limit: int = 2000, max_pages: int = 5) -> gpd.GeoDataFrame:
    """Fetch Mapillary image points inside the Madrid bounding box.

    Args:
        bbox: Tuple with south, west, north, east coordinates.
        token: Mapillary access token. If missing, returns an empty frame.
        limit: Page size requested from the API.
        max_pages: Maximum number of pages to fetch for reproducibility and API hygiene.

    Returns:
        GeoDataFrame with Mapillary image points.
    """
    cache_path = DATA_RAW / "mapillary_images_madrid.json"
    if cache_path.exists():
        payload = json.loads(cache_path.read_text(encoding="utf-8"))
    else:
        if not token:
            print("MAPILLARY_ACCESS_TOKEN is missing. Mapillary coverage will be empty.")
            return gpd.GeoDataFrame(columns=["source", "image_id", "captured_at", "geometry"], geometry="geometry", crs=WGS84)

        south, west, north, east = bbox
        url = "https://graph.mapillary.com/images"
        params = {
            "access_token": token,
            "bbox": f"{west},{south},{east},{north}",
            "fields": "id,geometry,captured_at",
            "limit": limit,
        }
        data = []
        page_number = 0
        while url and page_number < max_pages:
            response = requests.get(url, params=params, timeout=120)
            response.raise_for_status()
            page = response.json()
            data.extend(page.get("data", []))
            url = page.get("paging", {}).get("next")
            params = None
            page_number += 1
            time.sleep(0.5)
        payload = {"data": data}
        cache_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    rows = []
    for item in payload.get("data", []):
        coordinates = (item.get("geometry") or {}).get("coordinates")
        if not coordinates:
            continue
        lon, lat = coordinates
        rows.append({
            "source": "mapillary",
            "image_id": item.get("id"),
            "captured_at": item.get("captured_at"),
            "geometry": Point(lon, lat),
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)


mapillary_points = fetch_mapillary_images(MADRID_BBOX, MAPILLARY_TOKEN, max_pages=MAPILLARY_MAX_PAGES)
mapillary_points.to_parquet(DATA_PROCESSED / "mapillary_image_points.parquet", index=False)
mapillary_points.head()


## 5. Metricas de cobertura por zona

Se calculan conteos y densidades por km2 para cada atributo OSM y para imagenes Mapillary. Tambien se calcula riqueza de atributos: numero de atributos distintos presentes en cada zona.


In [ ]:
def count_points_by_zone(points: gpd.GeoDataFrame, zones: gpd.GeoDataFrame, value_column: str, prefix: str) -> pd.DataFrame:
    """Count point records per analysis zone.

    Args:
        points: Point GeoDataFrame in WGS84.
        zones: Polygon GeoDataFrame in WGS84.
        value_column: Column to split counts by.
        prefix: Prefix for generated count columns.

    Returns:
        DataFrame keyed by zone_id with count columns.
    """
    if points.empty:
        return pd.DataFrame({"zone_id": zones["zone_id"]})

    joined = gpd.sjoin(points, zones[["zone_id", "geometry"]], how="inner", predicate="within")
    counts = joined.groupby(["zone_id", value_column]).size().unstack(fill_value=0)
    counts = counts.add_prefix(prefix)
    return counts.reset_index()


zone_metrics = grid[["zone_id", "area_km2", "geometry"]].copy()

osm_counts = count_points_by_zone(osm_points, grid, "attribute", "osm_")
zone_metrics = zone_metrics.merge(osm_counts, on="zone_id", how="left")

if not mapillary_points.empty:
    mapillary_with_kind = mapillary_points.assign(kind="images")
    mapillary_counts = count_points_by_zone(mapillary_with_kind, grid, "kind", "mapillary_")
    zone_metrics = zone_metrics.merge(mapillary_counts, on="zone_id", how="left")
else:
    zone_metrics["mapillary_images"] = 0

count_columns = [column for column in zone_metrics.columns if column.startswith("osm_") or column.startswith("mapillary_")]
zone_metrics[count_columns] = zone_metrics[count_columns].fillna(0).astype(int)

osm_count_columns = [column for column in zone_metrics.columns if column.startswith("osm_")]
for column in count_columns:
    zone_metrics[f"{column}_per_km2"] = zone_metrics[column] / zone_metrics["area_km2"]

zone_metrics["osm_attribute_richness"] = (zone_metrics[osm_count_columns] > 0).sum(axis=1) if osm_count_columns else 0
zone_metrics["osm_total_accessibility_elements"] = zone_metrics[osm_count_columns].sum(axis=1) if osm_count_columns else 0
zone_metrics["mapillary_available"] = zone_metrics.get("mapillary_images", 0) > 0
zone_metrics["attributes_present"] = zone_metrics[osm_count_columns].gt(0).apply(lambda row: ", ".join([col.replace("osm_", "") for col, ok in row.items() if ok]), axis=1)
zone_metrics["attributes_absent"] = zone_metrics[osm_count_columns].eq(0).apply(lambda row: ", ".join([col.replace("osm_", "") for col, missing in row.items() if missing]), axis=1)

zone_metrics.head()


## 6. Puntuacion para seleccionar zona piloto

La puntuacion no mide accesibilidad real. Mide idoneidad para el MVP: zonas con suficientes datos OSM variados y cobertura visual Mapillary para inspeccion/evaluacion futura.

Criterios:

- Riqueza de atributos OSM.
- Densidad total de elementos OSM relacionados con accesibilidad.
- Cobertura Mapillary si hay token.
- Penalizacion suave por ausencia de atributos criticos.


In [ ]:
CRITICAL_ATTRIBUTES = ["osm_crossings", "osm_traffic_signals", "osm_tactile_paving", "osm_audible_signals", "osm_kerbs"]


def min_max(series: pd.Series) -> pd.Series:
    """Normalize a numeric series to 0..1, returning 0 when constant."""
    maximum = series.max()
    minimum = series.min()
    if maximum == minimum:
        return pd.Series(0.0, index=series.index)
    return (series - minimum) / (maximum - minimum)


zone_metrics["critical_attributes_present"] = sum(
    (zone_metrics[column] > 0).astype(int) for column in CRITICAL_ATTRIBUTES if column in zone_metrics.columns
)
zone_metrics["critical_attributes_missing"] = len([column for column in CRITICAL_ATTRIBUTES if column in zone_metrics.columns]) - zone_metrics["critical_attributes_present"]

zone_metrics["pilot_score"] = (
    0.40 * min_max(zone_metrics["osm_attribute_richness"])
    + 0.30 * min_max(zone_metrics["osm_total_accessibility_elements_per_km2"])
    + 0.20 * min_max(zone_metrics.get("mapillary_images_per_km2", pd.Series(0, index=zone_metrics.index)))
    + 0.10 * min_max(zone_metrics["critical_attributes_present"])
)

ranked_zones = zone_metrics.sort_values("pilot_score", ascending=False).copy()
ranked_zones[[
    "zone_id",
    "pilot_score",
    "osm_attribute_richness",
    "osm_total_accessibility_elements",
    "mapillary_images",
    "critical_attributes_present",
    "attributes_present",
    "attributes_absent",
]].head(20)


In [ ]:
metrics_csv = DATA_PROCESSED / "madrid_accessibility_density_by_zone.csv"
geojson_path = DATA_PROCESSED / "madrid_accessibility_density_by_zone.geojson"

zone_metrics.drop(columns="geometry").to_csv(metrics_csv, index=False)
zone_metrics.to_file(geojson_path, driver="GeoJSON")

print(f"Saved metrics to {metrics_csv}")
print(f"Saved GeoJSON to {geojson_path}")


## 7. Mapas de densidad

Se generan mapas HTML interactivos y PNG estaticos en `outputs/maps/`.


In [ ]:
def add_choropleth_layer(map_object: folium.Map, data: gpd.GeoDataFrame, column: str, layer_name: str) -> None:
    """Add a choropleth and tooltip layer to a Folium map."""
    folium.Choropleth(
        geo_data=json.loads(data.to_json()),
        data=data.drop(columns="geometry"),
        columns=["zone_id", column],
        key_on="feature.properties.zone_id",
        fill_color="YlOrRd",
        fill_opacity=0.65,
        line_opacity=0.25,
        legend_name=layer_name,
        name=layer_name,
    ).add_to(map_object)
    folium.GeoJson(
        data.to_json(),
        name=f"tooltips - {layer_name}",
        style_function=lambda _: {"fillOpacity": 0, "color": "#333333", "weight": 0.2},
        tooltip=GeoJsonTooltip(
            fields=["zone_id", column, "osm_attribute_richness", "mapillary_images", "attributes_present", "attributes_absent"],
            aliases=["Zona", column, "Riqueza OSM", "Imagenes Mapillary", "Presentes", "Ausentes"],
            sticky=False,
        ),
    ).add_to(map_object)


density_map = folium.Map(location=MADRID_CENTER, zoom_start=11, tiles="CartoDB positron")
add_choropleth_layer(density_map, zone_metrics, "pilot_score", "Puntuacion zona piloto")
add_choropleth_layer(density_map, zone_metrics, "osm_total_accessibility_elements_per_km2", "Densidad OSM accesibilidad")
if "mapillary_images_per_km2" in zone_metrics.columns:
    add_choropleth_layer(density_map, zone_metrics, "mapillary_images_per_km2", "Densidad Mapillary")
folium.LayerControl(collapsed=False).add_to(density_map)

density_map_path = MAPS_DIR / "madrid_accessibility_density_map.html"
density_map.save(density_map_path)
density_map


In [ ]:
plot_columns = ["pilot_score", "osm_attribute_richness", "osm_total_accessibility_elements_per_km2"]
if "mapillary_images_per_km2" in zone_metrics.columns:
    plot_columns.append("mapillary_images_per_km2")

fig, axes = plt.subplots(1, len(plot_columns), figsize=(6 * len(plot_columns), 6))
if len(plot_columns) == 1:
    axes = [axes]
for axis, column in zip(axes, plot_columns):
    zone_metrics.to_crs(MADRID_CRS).plot(column=column, ax=axis, legend=True, cmap="YlOrRd", edgecolor="white", linewidth=0.1)
    axis.set_title(column)
    axis.set_axis_off()
plt.tight_layout()
static_map_path = MAPS_DIR / "madrid_accessibility_density_static.png"
fig.savefig(static_map_path, dpi=180)
static_map_path


## 8. Lectura para decidir area piloto

Usar la tabla siguiente y el mapa `madrid_accessibility_density_map.html` para elegir zona piloto. Una buena zona piloto inicial deberia tener:

- Alto `pilot_score`.
- Varios atributos OSM presentes, no solo muchos elementos de un unico tipo.
- Mapillary disponible si se quiere inspeccion visual o VLM al final.
- Incertidumbre documentada: atributos criticos ausentes no bloquean el estudio, pero deben constar.
- Tamano manejable para revisar manualmente rutas de prueba.


In [ ]:
candidate_columns = [
    "zone_id",
    "pilot_score",
    "osm_attribute_richness",
    "osm_total_accessibility_elements_per_km2",
    "mapillary_images_per_km2" if "mapillary_images_per_km2" in zone_metrics.columns else "mapillary_images",
    "critical_attributes_present",
    "critical_attributes_missing",
    "attributes_present",
    "attributes_absent",
]
top_candidates = ranked_zones[candidate_columns].head(10)
top_candidates.to_csv(DATA_PROCESSED / "madrid_pilot_zone_candidates.csv", index=False)
top_candidates


## 9. Decision documentada

Completar despues de ejecutar el notebook:

- Zona piloto elegida:
- Motivos:
- Atributos disponibles:
- Atributos ausentes o poco densos:
- Riesgos de incertidumbre:
- Rutas de prueba propuestas:

Esta seccion debe copiarse o resumirse en `docs/journal.md` cuando exista.
